In [ ]:
import os
import base64
import pickle
import numpy as np
from io import BytesIO
import IPython
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FixedLocator, FixedFormatter
import pandas as pd
from datetime import datetime, timedelta
import yfinance as yf
import re

# =========================================================================
# CONFIGURATION
# =========================================================================
OUTPUT_PATH = "/content/drive/MyDrive/Stocks/output/"
HTML_FILE = os.path.join(OUTPUT_PATH, "Stock_report.html")
HTML = True

timeframes = [0.5, 1]

CACHE_FILE = "yfinance_market_cache.pkl"
CACHE_EXPIRATION_HOURS = 1

if os.path.exists(CACHE_FILE):
    os.remove(CACHE_FILE)

etfs = {
    # --- CORE ETFs ---
    "[CORE] Invesco NASDAQ-100 (25%)": "EQQQ.L",
    "[CORE] VanEck Semiconductor (40%)": "SMH",
    "[CORE] Vanguard FTSE Dev World (13%)": "V3AA.L",
    "[CORE] iShares S&P 500 Info Tech (22%)": "IUIT.L",

    # --- AI & ROBOTICS ETFs ---
    "[AI] ARK AI & Robotics": "ARKI.L",
    "[AI] Global X Robotics & AI": "BOTZ",
    "[AI] Robo Global Robotics & Auto": "ROBO",
    "[AI] VanEck Quantum Computing": "QNTM.L",
    "[AI] Xtrackers AI & Big Data": "XAIX.SW",

    # --- GENOMICS ETFs ---
    # "[GEN] ARK Genomic Revolution": "ARKG",
    # "[GEN] iShares Genomics Immunology": "IDNA",

    # --- SECTOR / THEMATIC ETFs ---
    # "[DEF] VanEck Defense ETF": "DFNS.L",
    # "[INF] Pacer Data & Digital Infra": "SRVR",
    # "[MED] VanEck Biotech (GLP-1 Alpha)": "BBH",

    # --- SPECULATIVE ETFs ---
    # "[SPEC] iShares MSCI South Korea": "EWY",
    # "[SPEC] First Trust Nasdaq Semi": "FTXL",
    # "[SPEC] Global X Hydrogen": "HYDR",
    # "[SPEC] CoinShares BTC Mining": "WGMI",
}

single_stocks = {
    # --- CORE stock ---
    "[CORE] Roche Holding AG (CHF) (100%)": "RO.SW",

    # --- [TECH] MEGA CAP / STANDARD STOCKS ---
    "[TECH] Alphabet Inc.": "GOOG",
    "[TECH] Amazon.com, Inc.": "AMZN",
    "[TECH] Apple Inc.": "AAPL",
    "[TECH] ASML Holding N.V.": "ASML",
    "[TECH] Intel Corporation": "INTC",
    "[TECH] Microsoft Corporation": "MSFT",
    "[TECH] NVIDIA Corporation": "NVDA",
    "[TECH] Oracle Corporation": "ORCL",
    "[TECH] Tesla, Inc.": "TSLA",
    "[TECH] Broadcom Inc.": "AVGO",
    "[TECH] Advanced Micro Devices, Inc.": "AMD",
    "[TECH] Palantir Technologies Inc.": "PLTR",
    "[TECH] Marvell Technology, Inc.": "MRVL",

    # --- [FIN] FINANCIALS / STABLE ---
    "[FIN] Circle Internet Group": "CRCL",

    # --- [ENG] ENERGY ---
    "[ENG] Chevron Corporation": "CVX",
    "[ENG] Bloom Energy Corporation": "BE",

    # --- [NUC] NUCLEAR STOCK PICKING BACKBONE ---
    "[NUC] Cameco Corporation": "CCJ",
    "[NUC] GE Vernova Inc.": "GEV",
    "[NUC] Sprott Physical Uranium": "SRUUF",
    "[NUC] Centrus Energy Corp.": "LEU",
    "[NUC] NuScale Power": "SMR",
    "[NUC] Oklo Inc.": "OKLO",

    # --- [QTM] PURE-PLAY QUANTUM COMPUTING BASKET ---
    "[QTM] IonQ, Inc.": "IONQ",
    "[QTM] D-Wave Quantum Inc.": "QBTS",
    "[QTM] Rigetti Computing, Inc.": "RGTI",
    "[QTM] Xanadu Quantum Technologies": "XNDU",
    "[QTM] Quantum Computing Inc.": "QUBT",
    "[QTM] Infleqtion": "INFQ",
    "[QTM] Horizon Quantum Computing": "HQ",
    "[QTM] Quantinuum": "QNT",

    # --- [CYBER] CYBERSECURITY SATELLITE ENGINE ---
    "[CYBER] CrowdStrike Holdings, Inc.": "CRWD",
    "[CYBER] Palo Alto Networks, Inc.": "PANW"
}

all_groups = {
    "ETFs": etfs,
    "Stocks": single_stocks
}

# =========================================================================
# THEMATIC STYLING DICTIONARY
# Note: Commenting out an asset in `etfs` or `single_stocks` automatically
# disables it entirely from the pipeline, ignoring its color here.
# =========================================================================
theme_styles = {
    # --- [NUC] NUCLEAR STOCK PICKING ---
    "[NUC] Cameco Corporation": {"color": "#D4AF37", "linestyle": "-"},
    "[NUC] GE Vernova Inc.": {"color": "#B8860B", "linestyle": "--"},
    "[NUC] Sprott Physical Uranium": {"color": "#FFD700", "linestyle": "-."},
    "[NUC] Centrus Energy Corp.": {"color": "#CD7F32", "linestyle": ":"},
    "[NUC] NuScale Power": {"color": "#FF8C00", "linestyle": "-"},
    "[NUC] Oklo Inc.": {"color": "#E31B23", "linestyle": "--"},

    # --- [QTM] QUANTUM STOCK PICKING ---
    "[QTM] IonQ, Inc.": {"color": "#800080", "linestyle": "-"},
    "[QTM] D-Wave Quantum Inc.": {"color": "#9370DB", "linestyle": "--"},
    "[QTM] Rigetti Computing, Inc.": {"color": "#8A2BE2", "linestyle": "-."},
    "[QTM] Quantum Computing Inc.": {"color": "#BA55D3", "linestyle": ":"},
    "[QTM] Xanadu Quantum Technologies": {"color": "#DA70D6", "linestyle": "-"},
    "[QTM] Infleqtion": {"color": "#4B0082", "linestyle": "--"},
    "[QTM] Horizon Quantum Computing": {"color": "#EE82EE", "linestyle": "-."},
    "[QTM] Quantinuum": {"color": "#46008B", "linestyle": ":"},

    # --- [CYBER] CYBERSECURITY ---
    "[CYBER] CrowdStrike Holdings, Inc.": {"color": "#000000", "linestyle": "-"},
    "[CYBER] Palo Alto Networks, Inc.": {"color": "#FF4500", "linestyle": "--"},

    # --- [CORE] CORE ETFs ---
    "[CORE] VanEck Semiconductor (40%)": {"color": "#00008B", "linestyle": "-"},
    "[CORE] Invesco NASDAQ-100 (25%)": {"color": "#4169E1", "linestyle": "--"},
    "[CORE] iShares S&P 500 Info Tech (22%)": {"color": "#00BFFF", "linestyle": "-."},
    "[CORE] Vanguard FTSE Dev World (13%)": {"color": "#4682B4", "linestyle": ":"},

    # --- [AI] AI & ROBOTICS ETFs ---
    "[AI] ARK AI & Robotics": {"color": "#800080", "linestyle": "-"},
    "[AI] Global X Robotics & AI": {"color": "#9370DB", "linestyle": "--"},
    "[AI] Robo Global Robotics & Auto": {"color": "#8A2BE2", "linestyle": "-."},
    "[AI] Xtrackers AI & Big Data": {"color": "#BA55D3", "linestyle": ":"},
    "[AI] VanEck Quantum Computing": {"color": "#DA70D6", "linestyle": "-"},

    # --- [TECH] MEGA CAP / STANDARD STOCKS ---
    "[TECH] NVIDIA Corporation": {"color": "#76B900", "linestyle": "-"},
    "[TECH] Microsoft Corporation": {"color": "#2F4F4F", "linestyle": "-"},
    "[TECH] Amazon.com, Inc.": {"color": "#FF8C00", "linestyle": "-"},
    "[TECH] Apple Inc.": {"color": "#708090", "linestyle": "-"},
    "[TECH] Alphabet Inc.": {"color": "#4A5D6E", "linestyle": "--"},
    "[TECH] Oracle Corporation": {"color": "#FF4500", "linestyle": "-"},
    "[TECH] ASML Holding N.V.": {"color": "#008080", "linestyle": "-"},
    "[TECH] Tesla, Inc.": {"color": "#E31B23", "linestyle": "--"},
    "[TECH] Intel Corporation": {"color": "#0071C5", "linestyle": "-."},
    "[TECH] Broadcom Inc.": {"color": "#CC0000", "linestyle": ":"},
    "[TECH] Advanced Micro Devices, Inc.": {"color": "#ED1C24", "linestyle": "-"},
    "[TECH] Palantir Technologies Inc.": {"color": "#3F4E4F", "linestyle": "--"},
    "[TECH] Marvell Technology, Inc.": {"color": "#00A3E0", "linestyle": "-."},

    # --- OTHER SECTOR STOCKS ---
    "[FIN] Circle Internet Group": {"color": "#A9A9A9", "linestyle": "-"},
    "[ENG] Chevron Corporation": {"color": "#8B4513", "linestyle": "-"},
    "[ENG] Bloom Energy Corporation": {"color": "#00FF00", "linestyle": "--"},
    "[HC] Roche Holding AG (CHF) (100%)": {"color": "#006400", "linestyle": "-."}
}

ticker_to_name = {**{v: k for k, v in etfs.items()}, **{v: k for k, v in single_stocks.items()}}
current_tickers = list(ticker_to_name.keys())

plot_data = {}
earliest_dates = {}
use_cache = False

growth_forecast_models = {
    "[CORE] Vanguard FTSE Dev World (13%)":  {"rate": 8.2,  "risk": "Low (High Conf)",   "cyclic": "",    "loss_risk": "Low"},
    "[CORE] Invesco NASDAQ-100 (25%)":      {"rate": 14.2, "risk": "Med-Low (Stable)",  "cyclic": "",    "loss_risk": "Low"},
    "[CORE] iShares S&P 500 Info Tech (22%)":{"rate": 15.6, "risk": "Medium (Stable)",   "cyclic": "",    "loss_risk": "Low"},
    "[CORE] VanEck Semiconductor (40%)":     {"rate": 18.2, "risk": "Med-High (Beta)",   "cyclic": "Yes", "loss_risk": "Low-Med"},

    "[AI] ARK AI & Robotics":                {"rate": 14.8, "risk": "Med-High (Beta)",   "cyclic": "",    "loss_risk": "Medium"},
    "[AI] Global X Robotics & AI":           {"rate": 15.1, "risk": "Med-High (Beta)",   "cyclic": "",    "loss_risk": "Medium"},
    "[AI] Robo Global Robotics & Auto":      {"rate": 12.9, "risk": "Medium (Stable)",   "cyclic": "",    "loss_risk": "Low"},
    "[AI] VanEck Quantum Computing":         {"rate": 24.5, "risk": "High (Speculative)","cyclic": "",    "loss_risk": "High"},
    "[AI] Xtrackers AI & Big Data":          {"rate": 19.2, "risk": "Med-High (Volatile)","cyclic": "",    "loss_risk": "Low-Med"},

    "[GEN] ARK Genomic Revolution":          {"rate": 11.5, "risk": "High (Volatile)",   "cyclic": "",    "loss_risk": "High"},
    "[GEN] iShares Genomics Immunology":     {"rate": 10.8, "risk": "High (Volatile)",   "cyclic": "",    "loss_risk": "High"},

    "[DEF] VanEck Defense ETF":              {"rate": 12.4, "risk": "Low-Med (Sovereign)","cyclic": "",    "loss_risk": "Low"},
    "[INF] Pacer Data & Digital Infra":      {"rate": 16.8, "risk": "Medium (Capex Hub)", "cyclic": "Yes", "loss_risk": "Low"},
    "[MED] VanEck Biotech (GLP-1 Alpha)":    {"rate": 14.5, "risk": "Medium (Secular)",   "cyclic": "",    "loss_risk": "Medium"},

    "[SPEC] CoinShares BTC Mining":          {"rate": 26.5, "risk": "Extreme (Volatile)", "cyclic": "Yes", "loss_risk": "Extreme"},
    "[SPEC] First Trust Nasdaq Semi":        {"rate": 19.0, "risk": "High (Beta)",        "cyclic": "Yes", "loss_risk": "Medium"},
    "[SPEC] Global X Hydrogen":              {"rate": 10.2, "risk": "High (Speculative)", "cyclic": "Yes", "loss_risk": "High"},
    "[SPEC] iShares MSCI South Korea":       {"rate": 9.5,  "risk": "Med-High (Geopol)",  "cyclic": "Yes", "loss_risk": "Med-High"}
}


# ==================================================================================================
# METHODOLOGY FOR COMPOUND ANNUAL GROWTH RATE (CAGR) FORECAST MODELS
# Framework: Forecasted CAGR = Base Index Return + Secular Alpha Premium - Risk/Cyclicality Discount
# ==================================================================================================

# 1. BASELINE ANCHORING (Historical Market Benchmarks)
# - Global Equity (~8.2%): Fundamental non-leveraged equity risk premium benchmark (VGWL).
# - Tech / Growth (~14.2% - 15.6%): 10-year baseline for mega-cap platform economics (EQQQ, IITU).

# 2. SECULAR ALPHA PREMIUMS (Forward-Looking Growth Catalysts)
# - Capex Infrastructure (+2.0% to +4.0%): Massive capital flows for hardware/data nodes (SMH, SRVR).
# - Product Alpha & Defense (+1.0% to +3.0%): Sovereign budgets & secular drug scaling (DFNS, SBIO).
# - S-Curve Adoption (+5.0% to +10.0%): Early-stage exponential disruption (AIXG, QGBL).

# 3. RISK, CYCLICALITY, AND CAPITAL INTENSITY DISCOUNTS
# - High-Beta Cyclicality (-1.5% to -4.0%): Smoothed for downcycles & inventory macro (SOXX, CSKR).
# - Capital Attrism / Burn (-2.0% to -5.0%): High cash-burn & regulatory hurdles (ARKG, HYDR).
# - Extreme Volatility Cap: Macro/halving constraints vs. historical mining peaks (WGMI).
# ==================================================================================================


stock_forecast_models = {
    # --- INDIVIDUAL STOCKS ---
    "[TECH] Alphabet Inc.":                 {"min_rate": 8.0,  "max_rate": 15.0, "risk": "Medium (Stable)", "cyclic": "No",  "loss_risk": "Low"},
    "[TECH] Amazon.com, Inc.":              {"min_rate": 10.0, "max_rate": 18.0, "risk": "Medium (Capex)",  "cyclic": "Yes", "loss_risk": "Low-Med"},
    "[TECH] Apple Inc.":                    {"min_rate": 5.0,  "max_rate": 12.0, "risk": "Low (Stable)",    "cyclic": "No",  "loss_risk": "Low"},
    "[TECH] ASML Holding N.V.":             {"min_rate": 12.0, "max_rate": 22.0, "risk": "Medium (Moat)",   "cyclic": "Yes", "loss_risk": "Medium"},
    "[TECH] Intel Corporation":             {"min_rate": 2.0,  "max_rate": 14.0, "risk": "High (Turnaround)","cyclic": "Yes", "loss_risk": "Medium"},
    "[TECH] Microsoft Corporation":         {"min_rate": 10.0, "max_rate": 16.0, "risk": "Low (Stable)",    "cyclic": "No",  "loss_risk": "Low"},
    "[TECH] NVIDIA Corporation":            {"min_rate": 15.0, "max_rate": 30.0, "risk": "High (Hardware)", "cyclic": "Yes", "loss_risk": "Medium"},
    "[TECH] Oracle Corporation":            {"min_rate": 8.0,  "max_rate": 15.0, "risk": "Medium (Cloud)",  "cyclic": "No",  "loss_risk": "Low-Med"},
    "[TECH] Tesla, Inc.":                   {"min_rate": 5.0,  "max_rate": 25.0, "risk": "High (Auto/AI)",  "cyclic": "Yes", "loss_risk": "High"},
    "[TECH] Broadcom Inc.":                 {"min_rate": 12.0, "max_rate": 20.0, "risk": "Low-Med (Moat)",  "cyclic": "No",  "loss_risk": "Low"},
    "[TECH] Advanced Micro Devices, Inc.":  {"min_rate": 10.0, "max_rate": 22.0, "risk": "High (Beta)",     "cyclic": "Yes", "loss_risk": "Medium"},
    "[TECH] Palantir Technologies Inc.":    {"min_rate": 14.0, "max_rate": 28.0, "risk": "High (Growth)",   "cyclic": "No",  "loss_risk": "Low-Med"},
    "[TECH] Marvell Technology, Inc.":      {"min_rate": 11.0, "max_rate": 24.0, "risk": "High (Electro)",  "cyclic": "Yes", "loss_risk": "Medium"},

    "[FIN] Circle Internet Group":          {"min_rate": 15.0, "max_rate": 35.0, "risk": "High (Crypto)",   "cyclic": "Yes", "loss_risk": "High"},
    "[ENG] Chevron Corporation":            {"min_rate": 4.0,  "max_rate": 10.0, "risk": "Med (Commodity)", "cyclic": "Yes", "loss_risk": "Medium"},
    "[ENG] Bloom Energy Corporation":       {"min_rate": 8.0,  "max_rate": 26.0, "risk": "High (Growth)",   "cyclic": "Yes", "loss_risk": "High"},
    "[HC] Roche Holding AG (CHF) (100%)":   {"min_rate": 5.0,  "max_rate": 10.0, "risk": "Low (Pharma)",    "cyclic": "No",  "loss_risk": "Low"},

    # --- NUC ---
    "[NUC] Cameco Corporation":      {"min_rate": 8.0,  "max_rate": 18.0, "risk": "Medium (Stable)", "cyclic": "Yes", "loss_risk": "Medium"},
    "[NUC] GE Vernova Inc.":         {"min_rate": 10.0, "max_rate": 20.0, "risk": "Medium (Capex)",  "cyclic": "Yes", "loss_risk": "Medium"},
    "[NUC] Sprott Physical Uranium": {"min_rate": 6.0,  "max_rate": 15.0, "risk": "Low-Med (Asset)", "cyclic": "Yes", "loss_risk": "Low"},
    "[NUC] Centrus Energy Corp.":    {"min_rate": 12.0, "max_rate": 25.0, "risk": "High (Geopol)",   "cyclic": "No",  "loss_risk": "High"},
    "[NUC] NuScale Power":           {"min_rate": -10.0,"max_rate": 35.0, "risk": "Extreme (Burn)",  "cyclic": "No",  "loss_risk": "Extreme"},
    "[NUC] Oklo Inc.":               {"min_rate": -15.0,"max_rate": 40.0, "risk": "Extreme (Burn)",  "cyclic": "No",  "loss_risk": "Extreme"},

    # --- QTM ---
    "[QTM] IonQ, Inc.":                  {"min_rate": -20.0, "max_rate": 45.0, "risk": "Extreme", "cyclic": "No", "loss_risk": "Extreme"},
    "[QTM] D-Wave Quantum Inc.":         {"min_rate": -25.0, "max_rate": 35.0, "risk": "Extreme", "cyclic": "No", "loss_risk": "Extreme"},
    "[QTM] Rigetti Computing, Inc.":     {"min_rate": -25.0, "max_rate": 30.0, "risk": "Extreme", "cyclic": "No", "loss_risk": "Extreme"},
    "[QTM] Xanadu Quantum Technologies": {"min_rate": -20.0, "max_rate": 50.0, "risk": "Extreme", "cyclic": "No", "loss_risk": "Extreme"},
    "[QTM] Quantum Computing Inc.":      {"min_rate": -30.0, "max_rate": 25.0, "risk": "Extreme", "cyclic": "No", "loss_risk": "Extreme"},
    "[QTM] Infleqtion":                  {"min_rate": -20.0, "max_rate": 45.0, "risk": "Extreme", "cyclic": "No", "loss_risk": "Extreme"},
    "[QTM] Horizon Quantum Computing":   {"min_rate": -20.0, "max_rate": 45.0, "risk": "Extreme", "cyclic": "No", "loss_risk": "Extreme"},
    "[QTM] Quantinuum":                  {"min_rate": -15.0, "max_rate": 40.0, "risk": "Extreme", "cyclic": "No", "loss_risk": "Extreme"},

    # --- CYBER ---
    "[CYBER] CrowdStrike Holdings, Inc.": {"min_rate": 12.0, "max_rate": 26.0, "risk": "Medium (Moat)", "cyclic": "No",  "loss_risk": "Low-Med"},
    "[CYBER] Palo Alto Networks, Inc.":   {"min_rate": 10.0, "max_rate": 22.0, "risk": "Low-Med (Stable)","cyclic": "No",  "loss_risk": "Low"}
}

HTML_TAG_COLORS = {
    "[CORE]": "#EBF4FA",
    "[AI]": "#F3E6F5",
    "[TECH]": "#E3F2FD",
    "[FIN]": "#ECEFF1",
    "[ENG]": "#FFF3E0",
    "[HC]": "#E8F5E9",
    "[NUC]": "#FFF8DC",
    "[QTM]": "#F8F8FF",
    "[CYBER]": "#FFEBEE"
}

def get_row_bg_color(asset_name):
    for tag, color in HTML_TAG_COLORS.items():
        if tag in asset_name:
            return color
    return "#FFFFFF"